# OT-2: Mean-Line Turbine Design

> Author: Elias Aoubala

> Date: 22/12/2025

In [ ]:
from turborocket.meanline.meanline_relations import TurbineStageDesign
from turborocket.fluids.fluids import IdealGas

import numpy as np
import pandas as pd

import yaml

## 1 - Background

This document encapsulates the mean-line design of the turbine-stage of the `OT-2: Man-Ray and the Dirty Bubble`. A high level mean line design study is conducted using our in-house python package `turboRocket`, from which an iterative optimisation is performed to evaluate for our key gas-generator mass flows needed to meet our power requirements.

## 2 - High Level Functional Parmaeters

The following high level functional parameters have been derived from the gas-generator CFD, along with the required shaft power, which has been illustrated in the following dictionary:

In [5]:
dic_gg = {
    "T_o": 868.1694130102591,               # Stagnation Temperature (K)
    "Cp": 2242.4288383607086,               # Specific Heat Capacity (J/kg K)
    "gamma": 1.290332008352664,             # Specific Heat Ratio (n.d.)
    "R": 504.5591863293963,                 # Specific Gas Constant (J/ kg K)
    "MR": np.float64(1.0),                  # Nominal Mixture Ratio of the Gas
    "P_cc": 2500000.0,                      # Combustion Chamber Pressure (Bar)
    "fu_stiffness": 1.0,                    # Fuel Stiffness Ratio
    "m_dot_f": 0.055977,                    # Fuel Mass Flow Rate (kg/s)
    "m_dot_o": 0.055977,                    # Oxidiser Mass Flow Rate (kg/s)
    "m_dot_t": 0.111954,                    # Total Mass Flow Rate (kg/s)
    "ox_stiffness": 1.0,                    # Oxidiser Injector Stiffness Ratio (n.d.)
}

dic_gg_geom = {
    "CdA_ox": np.float64(8.914471692165566e-07),
    "CdA_fu": np.float64(8.929216317561217e-07),
    "A_fu": np.float64(2.232304079390304e-06),
    "A_ox": np.float64(1.7828943384331133e-06),
    "Acc": 4.45357383e-05,
}

dic_rotor = {
    "N_shaft": 30e3,                        # Shaft Speed (RPM)
    "u_cis": 0.1,                           # 
    "Rt": 23,                               # Expansion Pressure Ratio (P_o/P_s nozzle exit)
    "phi_n": 0.85,                          # Loss Coefficient of Nozzle
    "alpha": 70,                            # Absolute Angle of Nozzle Exit
    "delta_r": 2e-3,                        # Tip Gap Height
    "N_nozzle": 8,                          # Number of Nozzles
    "N_rotor": 35,                          # Number of Rotor Blades
    "power": 24                             # Required Turbine Power (kW)
}

## 4 - Mean-Line Design Power Optimization

Our current approach for mean-line design does not design to user specification, but rather takes in some geometric parameters from the user, and calculates the expected turbine stage performance from it (in this case power, etc). As we have a target shaft power we must hit, this setions aims to conduct a primitive optimisation where the gas-generator mass flow is iterated on, until the target shaft power is met.

### 4.1 - Inital Parameter Guesses

As this problem is an optimisation problem, we will manually guess the blade spacing and thickness. These will updated once the turbine profile has been designed.

In [6]:
b = 15e-3
t = 5e-3

Based on these inputs, we can derive the required turbine power accordingly - using our refined loss model.

The main parameter we will iterate on the GG mass flow rate untill we reach our target power. We will do an adjoint-esque iterative study until the power requirement is reached.

We will instantiate an inital error of 0

In [7]:
error = 0

### 4.2 - Optimisation Loop for System Power

We will use a relaxation factor of **0.4** for the adjoint solver based on the relative error from the target power.

In [84]:
relax = 0.4

dm = dic_gg["m_dot_t"]*error*relax

dic_gg["m_dot_t"] -= dm

In [85]:
print(f"Blade Chord Length: {b*1e3}")
print(f"Blade Spacing: {t*1e3}")

Blade Chord Length: 15.0
Blade Spacing: 5.0


In [86]:
fluid = IdealGas(
    p=dic_gg["P_cc"],
    t=dic_gg["T_o"],
    gamma=dic_gg["gamma"],
    cp=dic_gg["Cp"],
    R=504,
)

stage = TurbineStageDesign(gas=fluid, m_dot=dic_gg["m_dot_t"], omega=dic_rotor["N_shaft"], alpha=(90 - dic_rotor["alpha"]))

stage.set_operating_point(u_cis=dic_rotor["u_cis"], Rt=dic_rotor["Rt"], b=b, t=t, delta_r=dic_rotor["delta_r"], N=dic_rotor["N_nozzle"])

result = stage.solve_performance(phi_n=dic_rotor["phi_n"])

Current Error: 9.49635259316943 %
Current Error: 0.789767861051967 %
Current Error: 0.07111373047558796 %
Current Error: 0.006359351721037127 %


We can get out power produced and calculate our error to repeat once again.

In [87]:
P = result["performance"]["Power"] * 1e-3

error = (P - dic_rotor["power"]) / dic_rotor["power"]

print(f"Relative Error: {error*1e2:.2f} %")

Relative Error: -0.00 %


**Final Gas Generator Mass Flow Rate**

In [88]:
%%render param

m_dot_t = dic_gg["m_dot_t"]*1e3 # g/s

<IPython.core.display.Latex object>

## 5 - Expected Mean-Line Performance Metrics

We can now get an idea of the specific power of the turbine and the associated performance Metrics

In [89]:
performance_dic = result["performance"]
performance_dic = {k: [v] for k, v in performance_dic.items()}
performance_df = pd.DataFrame(performance_dic)

pressure_dic = result["pressure"]
pressure_dic = {k: [v] for k, v in pressure_dic.items()}
pressure_df = pd.DataFrame(pressure_dic)

velocity_dic = result["velocity"]
velocity_dic = {k: [v] for k, v in velocity_dic.items()}
velocity_df = pd.DataFrame(velocity_dic)

temperature_dic = result["temperature"]
temperature_dic = {k: [v] for k, v in temperature_dic.items()}
temperature_df = pd.DataFrame(temperature_dic)

geometry_dic = result["geometry"]
geometry_dic = {k: [v] for k, v in geometry_dic.items()}
geometry_df = pd.DataFrame(geometry_dic)

mach_dic = result["mach"]
mach_dic = {k: [v] for k, v in mach_dic.items()}
mach_df = pd.DataFrame(mach_dic)

angles_dic = result["angles"]
angles_dic = {k: [v] for k, v in angles_dic.items()}
angles_df = pd.DataFrame(angles_dic)

### 5.1 - High Level Functional Parameters

In [90]:
performance_df["Power (kW)"] = performance_df["Power"].multiply(1e-3)
performance_df["Torque (Nm)"] = performance_df["Power"] / (25000 * 2 *np.pi/60)

performance_df

,dh,eps,phi_r,phi_l,m_leakage,eta_l,phi,eta_h,zeta_eps,eta_o,Power,Power (kW),Torque (Nm)
0,985358.187609,0.438324,0.788574,0.240595,0.031403,0.759405,0.85,0.248311,0.001967,0.186602,23999.372758,23.999373,9.167085


As can be seen, we are getting an efficiency on the order of **45%** with a specific power of 987 kW/kg/s

This works out to a GG size of the following:

### 5.2 - Expected Stage Pressures

These are the expected stage pressures in **Bar**.

In [91]:
pressure_df * 1e-5

,p_0,p_1,p_1o,p_1o_r,p_2o_r,p_2o
0,25.0,1.086957,8.201209,5.775221,2.840409,2.230187


### 5.3 - Expected Stage Temperatures

In [92]:
temperature_df

,t_0,t_1,t_1o_r,t_2,t_2o
0,868.169413,550.33941,802.294913,645.616452,758.936417


### 5.4 - Expected Stage Geometric Parameters

Here are the expected stage geometric parameters. We make a comparison between that calculated using the mean-line method, and that evaluated from our GG combustion solver to confirm the areas are comparable.

In [93]:
geometry_df["Nozzle Throat (mm)"] = [(dic_gg_geom["Acc"] / (dic_rotor["N_nozzle"]* np.pi))**(1/2) * 2 * 1e3]
geometry_df["Nozzle Exit (mm)"] = geometry_df["s_c"]*1e3
geometry_df["eps"] = geometry_df["A_1"] / geometry_df["A_0"]

throat_error = (geometry_df["A_0"][0] - dic_gg_geom["Acc"] )/ dic_gg_geom["Acc"]

print(f"Comparison of Throat Calculated using GG combustion solver: {throat_error*100:.1f} %")

geometry_df

Comparison of Throat Calculated using GG combustion solver: 16.5 %


,D_m,A_1,A_0,s_c,s_b,D_hub,D_tip,AR,Nozzle Throat (mm),Nozzle Exit (mm),eps
0,0.08937,0.000279,0.000052,0.006665,0.009165,0.080205,0.098535,0.611014,2.662345,6.665205,5.378913


### 5.5 - Expected Mach Numbers at Each Stage

In [94]:
mach_df

,m_star_c1,m_star_w1,m_star_w2,m_star_c2
0,1.699405,1.573971,1.241193,1.014735


### 5.6 - Expected Angles

For these angles, we subtracting from 90 to get the relative to the axial direction.

In [95]:
90 -angles_df

,beta_1,beta_2,alpha_2
0,67.409767,89.571414,54.071032


### 5.7 - Expected Stage Velocities

These are the expected absolute and relative velocities passing through the stage.

In [97]:
velocity_df

,u,c_1s,c_1,w_1,a_star_2,w_2,c_2,a_star_3
0,140.382206,1403.82206,1193.248751,1062.418113,674.99227,837.795822,712.503099,656.499616


## 6 - Exporting our Results

Now that we have done the high level mean-line design, we can extract our key parameters derived from the study, that we can place in a data fram and export as a yaml file, that can be read.

In [ ]:
results_dic = {**performance_dic, **pressure_dic, **velocity_dic, **temperature_dic, **geometry_dic, **mach_dic, **angles_dic}


